In [1]:
import pandas as pd
import numpy as np
import json
import os
import urllib3
import requests
import csv
import logging
import re
import shortuuid
from pathlib import Path
from datetime import datetime
from idmc_utils import generate_taskflow, package_import, upload_package
from urllib.parse import quote

# Set the pandas options
pd.set_option('display.max_columns', None)

# Initialise the job

In [2]:
# Initialise the log file
logging.basicConfig(
    filename='logs/console.log',
    level=logging.DEBUG,
    format='%(asctime)s:%(levelname)s:%(message)s',
)

In [3]:
# Read the config file
logging.info('Reading the config file')
with open('config/config.json', 'r') as infile:
    config = json.load(infile)


# Read the Inputs

In [4]:
# Read the execution plans
logging.info('Reading the execution plans')
dfPlans = pd.read_csv('in/plans.csv', dtype='str', encoding='utf-8', na_filter=False)
dfPlans['plan_step_order'] = dfPlans['plan_step_order'].astype(int)

# Convert the steps for task groups / sub taskflows
dfGroups = dfPlans[['plan_wid','group_name']].copy()
dfGroups = dfGroups.drop_duplicates()

dfPlans['plan_step_type'] = dfPlans.apply(lambda x: 'SUBTASKFLOW' if x['step_name'] in dfGroups[dfGroups['plan_wid'] == x['plan_wid']]['group_name'].unique() or x['plan_step_type'] == 'REGULAR' else x['plan_step_type'], axis=1) # 20261111 Updated filter to flag regular steps as sub-taskflows also
dfPlans['group_name'] = dfPlans['group_name'].fillna('')
dfPlans = dfPlans[dfPlans['group_name'] == '']
dfPlans['plan_step_wid'] = dfPlans.apply(lambda x: '' if x['plan_step_type'] == 'SUBTASKFLOW' else x['plan_step_wid'], axis=1)
dfPlans['step_wid'] = dfPlans.apply(lambda x: '' if x['plan_step_type'] == 'SUBTASKFLOW' else x['step_wid'], axis=1)
dfPlans['step_cmd'] = dfPlans.apply(lambda x: '' if x['plan_step_type'] == 'SUBTASKFLOW' else x['step_cmd'], axis=1)
dfPlans = dfPlans.drop_duplicates()

#TODO remove the below filter once done with testing
test_plans = [
    'Echo Employee Snapshot Oracle R12.1.3'
    #, 
    #'Echo GL Balance Refresh', 
    #'Echo Account Dimension', 
    #'Human Resources - Oracle R1213 - Flexfield', 
    #'Common-LoadDayDimension Universal',
    #'Process_Quality_OracleR1213'
    #'Financials_Universal'
    ]
dfPlans = dfPlans[dfPlans['plan_name'].isin(test_plans)]


In [5]:
# Read the execution plans
logging.info('Reading the sub-execution plans')
dfSubPlans = pd.read_csv('in/sub-plans.csv', dtype='str', encoding='utf-8', na_filter=False)
dfSubPlans['plan_step_order'] = dfSubPlans['plan_step_order'].astype(int)

#TODO remove the below filter once done with testing
#test_plans = [
#    'TASK_GROUP_Extract_EmployeeDailySnapshotFact_Performance_Review'
#    ]
#dfSubPlans = dfSubPlans[dfSubPlans['plan_name'].isin(test_plans)]


In [6]:
# Read the parameter values file
dfParamVals = pd.read_csv('in/param_values.csv', dtype='str', encoding='utf-8', na_filter=False)
dfParamVals['sequence'] = dfParamVals['sequence'].astype(float)
dfParamVals = dfParamVals.sort_values(['obj_wid', 'sequence'])

# String join the multi line values
dfParamVals = dfParamVals.groupby(['obj_wid','obj_type'])
dfParamVals = dfParamVals['textfield'].apply(lambda x: ''.join(x))
dfParamVals = pd.DataFrame(dfParamVals)
dfParamVals = dfParamVals.reset_index()
dfParamVals['param_values'] = dfParamVals['textfield'].str.split('_dac_sep_')
dfParamVals = dfParamVals.explode('param_values')
dfParamVals['value_type'] = dfParamVals['param_values'].str.extract(r'^(\S+)=')
dfParamVals['value'] = dfParamVals['param_values'].str.extract(r'^\S+=(.*)$')
dfParamVals = dfParamVals[~dfParamVals['value_type'].isna()]
dfParamVals = dfParamVals.reset_index(drop=True)
dfParamVals = dfParamVals.pivot(index=['obj_wid', 'obj_type', 'textfield'], columns=['value_type'], values=['value'])
dfParamVals = dfParamVals.reset_index()
dfParamVals.columns = ['_'.join(col).strip() for col in dfParamVals.columns.values]
dfParamVals['is_static'] = dfParamVals['value_Runtime'].apply(lambda x: False if x == 'true' else True)
dfParamVals = dfParamVals.rename(columns={'obj_wid_': 'param_wid', 'obj_type_': 'param_type', 'textfield_': 'raw_text'})


In [7]:
# Read the task and src system parameter files
dfTaskSrcParams = pd.read_csv('in/params-task-src-sys.csv', dtype='str', encoding='utf-8', na_filter=False)
dfTaskSrcParams = dfTaskSrcParams.merge(dfPlans, how='inner', on=['step_wid'])
dfTaskSrcParams = dfTaskSrcParams[['name', 'value', 'type_cd', 'step_wid', 'datatype', 'context_type', 'inactive_flag', 'comments', 'plan_name', 'step_name', 'param_wid']]
dfTaskSrcParams = dfTaskSrcParams.drop_duplicates()
#dfTaskSrcParams['param_filename'] = dfTaskSrcParams['plan_name'].apply(lambda x: re.sub(r'\W+', '_', x) + '.param')

# Join the values onto the params
dfTaskSrcParams = dfTaskSrcParams.merge(dfParamVals, how='left', on='param_wid')

In [8]:
# Read the execution parameter files
dfExecParams = pd.read_csv('in/params-exec.csv', dtype='str', encoding='utf-8', na_filter=False)
dfExecParams = dfExecParams.merge(dfPlans, how='left', on='plan_wid')
dfExecParams = dfExecParams[['param_wid', 'name', 'type_cd', 'plan_wid', 'datatype', 'context_type',
       'inactive_flag', 'comments', 'plan_name']]
dfExecParams = dfExecParams.drop_duplicates()
#dfExecParams['param_filename'] = dfExecParams['plan_name'].apply(lambda x: re.sub(r'\W+', '_', x) + '.param')
dfExecParams['name'] = dfExecParams['name'].apply(lambda x: x if x.startswith('$$') else '$$' + x if len(x) > 0 else x)

# Join the values onto the params
dfExecParams = dfExecParams.merge(dfParamVals, how='left', on='param_wid')

# Lookup the Converted Mapping Task IDs

In [9]:
# Login to IDMC
logging.info('Logging into IDMC')

data = {
    'username': config['idmc']['user'],
    'password': config['idmc']['password']
}

url = 'https://' + config['idmc']['host'] + '/saas/public/core/v3/login'
r = requests.post(url=url, json=data)
    
# Convert the response into a datframe
try:
    result = r.json()
    sessionID = result['userInfo']['sessionId']
except Exception as e:
    print(e)
    print(r.data)

In [10]:
# Get the secure agent ID
logging.info('Getting the secure agent ID')
url = 'https://' + config['idmc']['pod'] + '.' + config['idmc']['host'] + '/saas/api/v2/runtimeEnvironment/name/' + quote(config['idmc']['agentGroupName'])
headers = { 'Accept': 'application/json', 'icSessionId': sessionID }
r = requests.get(url=url, headers=headers)
    
# Convert the response into a datframe
try:
    result = r.json()
    agentGroupID = result['id']
    agentGroupGUID = result['federatedId']
    agentGroupName = config['idmc']['agentGroupName']
except Exception as e:
    print(e)
    print(r.data)


In [11]:
# Initialise the tasks data frame
logging.info('Getting a list of the mapping tasks')
limit = 200
i = 0
dfTasks = pd.DataFrame()

# Page through mapping task queries
while True:

    # Get a list of the mapping tasks
    url = 'https://' + config['idmc']['pod'] + '.' + config['idmc']['host'] + '/saas/public/core/v3/objects'
    params = {
        'q': 'type=="MTT" and location=="' + config['idmc']['folderPath'] + '"',
        'limit': limit,
        'skip': i
    }
    headers={ 'Accept': 'application/json', 'INFA-SESSION-ID': sessionID }
    r = requests.get(url=url, params=params, headers=headers)
        
    # Convert the response into a datframe
    result = r.json()
    dfTmp = pd.json_normalize(result)
    dfResp = dfTmp.copy()
    taskCount = dfResp.iloc[0]['count']
    dfResp = dfResp['objects'].explode()
    dfResp = pd.DataFrame(dfResp)
    dfResp = pd.json_normalize(dfResp['objects'])
    dfTasks = pd.concat([dfTasks, dfResp], ignore_index=True)

    # Break if all records have been returned
    i = i + limit
    if i > taskCount:
        print(f'Finished. dfCount = { len(dfTasks) }; i = { i }; taskCount = { taskCount }')
        break

    print(f'Looping. dfCount = { len(dfTasks) }; i = { i }; taskCount = { taskCount }')

# Clean up the list of returned tasks
dfTasks = dfTasks.drop(columns=['tags'])
dfTasks = dfTasks.drop_duplicates()



Looping. dfCount = 200; i = 200; taskCount = 547
Looping. dfCount = 400; i = 400; taskCount = 547
Finished. dfCount = 547; i = 600; taskCount = 547


In [12]:
# Standardise the task data
dfTasks['step_name'] = dfTasks['path'].apply(lambda x: os.path.basename(x))
dfTasks = dfTasks[['step_name','id','path']]
dfTasks = dfTasks.rename(columns={'id': 'infa_id', 'path': 'infa_path'})

# Join onto the execution plans
dfPlans = dfPlans.merge(dfTasks, how='left', on='step_name')
dfPlans['agent_id'] = agentGroupID
dfPlans['agent_guid'] = agentGroupGUID
dfPlans['agent_name'] = agentGroupName
dfPlans['script_dir'] = config['local']['scriptsDir']
dfPlans['script_args'] = '' # TODO placeholder for any args that need to be passed to the step script

# Join onto the sub-execution plans
dfSubPlans = dfSubPlans.merge(dfTasks, how='left', on='step_name')
dfSubPlans['agent_id'] = agentGroupID
dfSubPlans['agent_guid'] = agentGroupGUID
dfSubPlans['agent_name'] = agentGroupName
dfSubPlans['script_dir'] = config['local']['scriptsDir']
dfSubPlans['script_args'] = '' # TODO placeholder for any args that need to be passed to the step script

In [13]:
# Log an error if any plans did not find an existing matching task
dfMissing = dfPlans[(dfPlans['infa_id'] == '') | (dfPlans['infa_id'].isna())].copy()
if len(dfMissing.index) > 0:
    logging.error('Some plans are missing a converted mapping task. Please see "out/missing_tasks.csv" for more details')
    dfMissing.to_csv('out/missing_tasks.csv', index=False, quoting=csv.QUOTE_ALL)

# Generate the Parameter Files

In [14]:
logging.info('Generating the parameter files...')

# Set the param file header
lines = ['#USE_SECTIONS', '']

# Set the taskflow level parameters
for idx, row in dfExecParams.iterrows():
    plan_name = re.sub(r'\W+', '_', row['plan_name'])
    param_name = row['name']

    lines.append(f'[Default].[{ plan_name }]')

    if row['is_static'] and row['datatype'] == 'Text':
        param_value = row['value_Text']
    elif row['is_static'] and row['datatype'] == 'Timestamp' and row['value_Function'] == 'TO_CUSTOM':
        param_value = row['value_Value']
        date_part, time_part = param_value.split(' at ')
        time_part_fixed = ':'.join(time_part.split(':')[:3])
        datetime_str = f"{date_part} {time_part_fixed}"
        dt = datetime.strptime(datetime_str, "%Y%m%d %H:%M:%S")
        param_value = dt.strftime("%Y-%m-%dT%H:%M:%S")
    elif row['datatype'] == 'SQL':
        param_value = row['value_SQL']
    else:
        param_value = row['value']
    lines.append(f"{ param_name }={ param_value }")
    lines.append('')

# Set the taskflow mapping task level parameters
for idx, row in dfTaskSrcParams.iterrows():
    plan_name = re.sub(r'\W+', '_', row['plan_name'])
    step_name = row['step_name']
    param_name = row['name']

    lines.append(f'[Default].[{ plan_name }].[{ step_name }]')

    if row['is_static'] and row['datatype'] == 'Text':
        param_value = row['value_Text']
    elif row['is_static'] and row['datatype'] == 'Timestamp' and row['value_Function'] == 'TO_CUSTOM':
        param_value = row['value_Value']
        if pd.notna(param_value) and ' at ' in param_value:
            date_part, time_part = param_value.split(' at ')
            time_part_fixed = ':'.join(time_part.split(':')[:3])
            datetime_str = f"{date_part} {time_part_fixed}"
            dt = datetime.strptime(datetime_str, "%Y%m%d %H:%M:%S")
            param_value = dt.strftime("%Y-%m-%dT%H:%M:%S")
        else:
            param_value = ''
    elif row['datatype'] == 'SQL':
        param_value = row['value_SQL']
    else:
        param_value = row['value']
    lines.append(f"{ param_name }={ param_value }")
    lines.append('')

lines.append('')

# Save the parameter file
with open(f'out/{ config['local']['paramFile'] }', 'w', encoding='utf-8') as file:
    file.write('\n'.join(lines))

# Generate the Task Group Sub-Taskflows

In [15]:
logging.info('Generating the sub-taskflows...')

# Filter out any missing tasks
dfSubPlansFil = dfSubPlans.copy()
dfSubPlansFil = dfSubPlansFil[(dfSubPlansFil['infa_id'] != '') & (dfSubPlansFil['infa_id'].notna())]

planIds = dfSubPlansFil['plan_wid'].unique()

for planId in planIds:

    # Get the plan for the current id
    dfPlan = dfSubPlansFil[dfSubPlansFil['plan_wid'] == planId].copy()

    # Generate the unique identifiers
    dfPlan['dac2idmc_step_id'] = dfPlan.apply(lambda x: "task" + shortuuid.uuid()[:8], axis=1)
    map_order_uuid = { step_order: "group" + shortuuid.uuid()[:8] for step_order in dfPlan['plan_step_order'].unique() }
    dfPlan = dfPlan.sort_values(['plan_wid', 'plan_step_order', 'step_name'])
    dfPlan['sequence'] = range(len(dfPlan))
    dfPlan['dac2idmc_group_id'] = dfPlan['plan_step_order'].map(map_order_uuid)
    dfPlan['dac2idmc_next_id'] = dfPlan['dac2idmc_step_id'].shift(-1)

    # Get the next group IDs and sort the plans
    dfGroups = dfPlan[['plan_step_order','dac2idmc_group_id']].copy()
    dfGroups = dfGroups.drop_duplicates()
    dfGroups = dfGroups.sort_values(['plan_step_order'])
    dfGroups['dac2idmc_next_group'] = dfGroups['dac2idmc_group_id'].shift(-1)
    dfGroups = dfGroups[['dac2idmc_group_id','dac2idmc_next_group']]

    dfPlan = dfPlan.merge(dfGroups, how='inner', on='dac2idmc_group_id')
    dfPlan = dfPlan.sort_values(['sequence'])

    #TODO Uncomment the below lines for troubleshooting
    #print(planId)
    #dfPlan.to_excel(f'out/{ planId }.xlsx')

    # Generate the taskflow ID
    taskflowID = shortuuid.uuid()
    taskflowName = dfPlan.iloc[0]['plan_name']
    taskflowName = re.sub(r'\W+', '_', taskflowName)

    logging.info(f'Create workspace tree "tmp/{ taskflowName }/Explore/Default"')

    # Create the workspace directories
    treePath = Path(f'tmp/{ taskflowName }/Explore/Default')
    treePath.mkdir(parents=True, exist_ok=True)

    logging.info(f'Generating taskflow "{ taskflowName }"')
    generate_taskflow.generate_taskflow(taskflowID, taskflowName, dfPlan, config)

    logging.info(f'Packaging taskflow "{ taskflowName }"')
    package_import.package_import(taskflowID, taskflowName, dfPlan)
    logging.info(f'Uploading the package "out/{ taskflowName }.zip"...')
    upload_package.upload_package(sessionID, taskflowName, config)

    logging.info(f'Done!')

logging.info('All sub-taskflows have been generated!')

{"jobId":"dQCAUfziTcohg2etCobAAN","jobStatus":{"state":"NOT_STARTED","message":null},"checksumValid":true}
{"id":"dQCAUfziTcohg2etCobAAN","createTime":"2026-11-11T02:51:40.000Z","updateTime":"2026-11-11T02:51:41.053Z","name":"TASK_GROUP_Extract_SpecificationTest_Dimension-MsbA4L9r","startTime":"2026-11-11T02:51:40.895Z","endTime":null,"status":{"state":"IN_PROGRESS","message":"In Progress"},"objects":null,"sourceOrgId":"2lh8CM1r57OkWgGofQVMIk","checksumValid":true}
{"id":"dQCAUfziTcohg2etCobAAN","createTime":"2026-11-11T02:51:40.000Z","updateTime":"2026-11-11T02:51:41.000Z","name":"TASK_GROUP_Extract_SpecificationTest_Dimension-MsbA4L9r","startTime":"2026-11-11T02:51:41.000Z","endTime":"2026-11-11T02:51:41.000Z","status":{"state":"SUCCESSFUL","message":"Import completed successfully"},"objects":null,"sourceOrgId":"2lh8CM1r57OkWgGofQVMIk","checksumValid":true}
{"jobId":"18txf6sCeLGbJUheSmdVmQ","jobStatus":{"state":"NOT_STARTED","message":null},"checksumValid":true}
{"id":"18txf6sCeLGbJU

# Lookup Sub-Taskflow ID's

In [16]:
# Initialise the tasks data frame
logging.info('Getting a list of the subtaskflows')
limit = 200
i = 0
dfTaskflows = pd.DataFrame()

# Page through mapping task queries
while True:

    # Get a list of the mapping tasks
    url = 'https://' + config['idmc']['pod'] + '.' + config['idmc']['host'] + '/saas/public/core/v3/objects'
    params = {
        'q': 'type=="TASKFLOW" and location=="' + config['idmc']['folderPath'] + '"',
        'limit': limit,
        'skip': i
    }
    headers={ 'Accept': 'application/json', 'INFA-SESSION-ID': sessionID }
    r = requests.get(url=url, params=params, headers=headers)
        
    # Convert the response into a datframe
    result = r.json()
    dfTmp = pd.json_normalize(result)
    dfResp = dfTmp.copy()
    taskCount = dfResp.iloc[0]['count']
    dfResp = dfResp['objects'].explode()
    dfResp = pd.DataFrame(dfResp)
    dfResp = pd.json_normalize(dfResp['objects'])
    dfTaskflows = pd.concat([dfTaskflows, dfResp], ignore_index=True)

    # Break if all records have been returned
    i = i + limit
    if i > taskCount:
        print(f'Finished. dfCount = { len(dfTaskflows) }; i = { i }; taskCount = { taskCount }')
        break

    print(f'Looping. dfCount = { len(dfTaskflows) }; i = { i }; taskCount = { taskCount }')

# Clean up the list of returned tasks
dfTaskflows = dfTaskflows.drop(columns=['tags'])
dfTaskflows = dfTaskflows.drop_duplicates()
dfTaskflows['taskflow_name'] = dfTaskflows['path'].apply(lambda x: os.path.basename(x))
dfTaskflows = dfTaskflows[['taskflow_name','id','path']]
dfTaskflows = dfTaskflows.rename(columns={'id': 'infa_id', 'path': 'infa_path'})


Finished. dfCount = 64; i = 200; taskCount = 64


In [17]:
# Lookup the sub-taskflow id and path info
dfPlans['infa_id'] = dfPlans.apply(lambda x: dfTaskflows[dfTaskflows['taskflow_name'] == x['step_name']].iloc[0]['infa_id'] if x['plan_step_type'] == 'SUBTASKFLOW' else x['infa_id'], axis=1)
dfPlans
dfPlans['infa_path'] = dfPlans.apply(lambda x: dfTaskflows[dfTaskflows['taskflow_name'] == x['step_name']].iloc[0]['infa_path'] if x['plan_step_type'] == 'SUBTASKFLOW' else x['infa_path'], axis=1)

# Generate the Execution Plan Taskflows

In [18]:
logging.info('Generating the taskflows...')

# Filter out any missing tasks
dfPlansFil = dfPlans.copy()
dfPlansFil = dfPlansFil[(dfPlansFil['infa_id'] != '') & (dfPlansFil['infa_id'].notna())]

dfTemp = dfPlansFil.copy()

#TODO Remove the below filter after testing is finished
#dfPlansFil = dfPlansFil[dfPlansFil['step_name']== 'SDE_Universal_APTermsDimension']

planIds = dfPlansFil['plan_wid'].unique()

for planId in planIds:

    # Get the plan for the current id
    dfPlan = dfPlansFil[dfPlansFil['plan_wid'] == planId].copy()

    # Generate the unique identifiers
    dfPlan['dac2idmc_step_id'] = dfPlan.apply(lambda x: "task" + shortuuid.uuid()[:8], axis=1)
    map_order_uuid = { step_order: "group" + shortuuid.uuid()[:8] for step_order in dfPlan['plan_step_order'].unique() }
    dfPlan = dfPlan.sort_values(['plan_wid', 'plan_step_order', 'step_name'])
    dfPlan['sequence'] = range(len(dfPlan))
    dfPlan['dac2idmc_group_id'] = dfPlan['plan_step_order'].map(map_order_uuid)
    dfPlan['dac2idmc_next_id'] = dfPlan['dac2idmc_step_id'].shift(-1)

    # Get the next group IDs and sort the plans
    dfGroups = dfPlan[['plan_step_order','dac2idmc_group_id']].copy()
    dfGroups = dfGroups.drop_duplicates()
    dfGroups = dfGroups.sort_values(['plan_step_order'])
    dfGroups['dac2idmc_next_group'] = dfGroups['dac2idmc_group_id'].shift(-1)
    dfGroups = dfGroups[['dac2idmc_group_id','dac2idmc_next_group']]

    dfPlan = dfPlan.merge(dfGroups, how='inner', on='dac2idmc_group_id')
    dfPlan = dfPlan.sort_values(['sequence'])

    #TODO Uncomment the below lines for troubleshooting
    #print(planId)
    #dfPlan.to_excel(f'out/{ planId }.xlsx')

    # Generate the taskflow ID
    taskflowID = shortuuid.uuid()
    taskflowName = dfPlan.iloc[0]['plan_name']
    taskflowName = re.sub(r'\W+', '_', taskflowName)

    logging.info(f'Create workspace tree "tmp/{ taskflowName }/Explore/Default"')

    # Create the workspace directories
    treePath = Path(f'tmp/{ taskflowName }/Explore/Default')
    treePath.mkdir(parents=True, exist_ok=True)

    logging.info(f'Generating taskflow "{ taskflowName }"')
    generate_taskflow.generate_taskflow(taskflowID, taskflowName, dfPlan, config)

    logging.info(f'Packaging taskflow "{ taskflowName }"')
    package_import.package_import(taskflowID, taskflowName, dfPlan)
    logging.info(f'Uploading the package "out/{ taskflowName }.zip"...')
    upload_package.upload_package(sessionID, taskflowName, config)

    logging.info(f'Done!')

logging.info('All taskflows have been generated!')

{"jobId":"fhHxwY4HqqOkmZoGMamhyf","jobStatus":{"state":"NOT_STARTED","message":null},"checksumValid":true}
{"id":"fhHxwY4HqqOkmZoGMamhyf","createTime":"2026-11-11T02:52:44.000Z","updateTime":"2026-11-11T02:52:44.623Z","name":"Echo_Employee_Snapshot_Oracle_R12_1_3-JQp6iJWK","startTime":"2026-11-11T02:52:44.485Z","endTime":null,"status":{"state":"IN_PROGRESS","message":"In Progress"},"objects":null,"sourceOrgId":"2lh8CM1r57OkWgGofQVMIk","checksumValid":true}
{"id":"fhHxwY4HqqOkmZoGMamhyf","createTime":"2026-11-11T02:52:44.000Z","updateTime":"2026-11-11T02:52:45.000Z","name":"Echo_Employee_Snapshot_Oracle_R12_1_3-JQp6iJWK","startTime":"2026-11-11T02:52:44.000Z","endTime":"2026-11-11T02:52:45.000Z","status":{"state":"SUCCESSFUL","message":"Import completed successfully"},"objects":null,"sourceOrgId":"2lh8CM1r57OkWgGofQVMIk","checksumValid":true}


# Testing below this line